In [1]:
import warnings
warnings.filterwarnings('ignore')

import os.path as osp
import torch
import torch.nn.functional as F
import torch.nn as nn
import time
import copy

import torch_geometric.transforms as T
from torch_geometric.datasets import Planetoid
from torch_geometric.nn import SplineConv
from torch_geometric.typing import WITH_TORCH_SPLINE_CONV

import torch.nn.utils.prune as prune
from torch.nn.utils.prune import global_unstructured, L1Unstructured


In [2]:
def get_num_parameters(model: nn.Module, count_nonzero_only=False) -> int:
    """
    calculate the total number of parameters of model
    :param count_nonzero_only: only count nonzero weights
    """
    num_counted_elements = 0
    for param in model.parameters():
        if count_nonzero_only:
            num_counted_elements += param.count_nonzero()
        else:
            num_counted_elements += param.numel()
    return num_counted_elements


def get_model_size(model: nn.Module, data_width=32, count_nonzero_only=True) -> int:
    """
    calculate the model size in bits
    :param data_width: #bits per element
    :param count_nonzero_only: only count nonzero weights
    """
    return get_num_parameters(model, count_nonzero_only) * data_width

Byte = 8
KiB = 1024 * Byte
MiB = 1024 * KiB
GiB = 1024 * MiB

  

In [3]:
import warnings
warnings.filterwarnings('ignore')
import sys, os
sys.path.append(os.path.abspath("../"))

import torch


## Loading Dataset

In [4]:

if not WITH_TORCH_SPLINE_CONV:
    quit("This example requires 'torch-spline-conv'")

dataset = 'Cora'
transform = T.Compose([
    T.RandomNodeSplit(num_val=500, num_test=500),
    T.TargetIndegree(),
])
path =  'data'
dataset = Planetoid(path, dataset, transform=transform)
data = dataset[0]


# Model

In [5]:
class Net(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = SplineConv(dataset.num_features, 16, dim=1, kernel_size=2)
        self.conv2 = SplineConv(16, dataset.num_classes, dim=1, kernel_size=2)

    def forward(self):
        x, edge_index, edge_attr = data.x, data.edge_index, data.edge_attr
        x = F.dropout(x, training=self.training)
        x = F.elu(self.conv1(x, edge_index, edge_attr))
        x = F.dropout(x, training=self.training)
        x = self.conv2(x, edge_index, edge_attr)
        return F.log_softmax(x, dim=1)


device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model, data = Net().to(device), data.to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.01, weight_decay=5e-3)

In [6]:


def train(l2):
    model.train()
    optimizer.zero_grad()
    loss=F.nll_loss(model()[data.train_mask], data.y[data.train_mask])
    l2_reg = torch.tensor(0.)
    for param in model.parameters():
        l2_reg += torch.norm(param)
        
    # Combine the loss function with L2 regularization
    loss += (l2 * l2_reg)
    loss.backward() 
    
    optimizer.step()
    return loss





@torch.no_grad()
def test(model):
    model.eval()
    log_probs, accs = model(), []
    for _, mask in data('train_mask', 'test_mask'):
        pred = log_probs[mask].max(1)[1]
        acc = pred.eq(data.y[mask]).sum().item() / mask.sum().item()
        accs.append(acc)
    return accs
   



In [8]:
l2_lambda = 0

for epoch in range(10):
    loss_trn=train( l2_lambda)
    
    #Report
    train_acc, test_acc = test(model)
    #if epoch % 10 == 0:
    print(f'Epoch: {epoch:03d},Train loss:{loss_trn:.4f}, Val: {train_acc:.4f}, Test: {test_acc:.4f}')

Epoch: 000,Train loss:1.9549, Val: 0.3653, Test: 0.3580
Epoch: 001,Train loss:1.7463, Val: 0.4268, Test: 0.4040
Epoch: 002,Train loss:1.5684, Val: 0.5205, Test: 0.4960
Epoch: 003,Train loss:1.4131, Val: 0.6446, Test: 0.6080
Epoch: 004,Train loss:1.2709, Val: 0.7722, Test: 0.7520
Epoch: 005,Train loss:1.1183, Val: 0.8630, Test: 0.8280
Epoch: 006,Train loss:0.9740, Val: 0.8964, Test: 0.8740
Epoch: 007,Train loss:0.8410, Val: 0.9052, Test: 0.8900
Epoch: 008,Train loss:0.7563, Val: 0.9069, Test: 0.8880
Epoch: 009,Train loss:0.6615, Val: 0.9110, Test: 0.8900


## Manual Measurement

In [7]:
import statistics as stat

l2_lambda =0.0001
Eva_final=dict()
num_epoch=50


Accuracy=[]
T_inference=[]
Num_parm=[]
Model_size=[]

In [8]:
for i in range(3):  
    print('_________________________________________________________')
    print(f'This is step:{i+1}')
      
    Eva=dict()
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model, data = Net().to(device), data.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.01, weight_decay=5e-3)

    for epoch in range(1,num_epoch):
        loss_trn=train( l2_lambda)
        #Report
        train_acc, test_acc = test(model)
        if epoch % 10 == 0:
            print(f'Epoch: {epoch:03d},Train loss:{loss_trn:.4f}, Val: {train_acc:.4f}, Test: {test_acc:.4f}')

    t0=time.time()
    val_acc,test_acc=test(model)
    t1=time.time()
    t_inference=t1 - t0  
    model_size = get_model_size(model,count_nonzero_only=True)
    num_parm=get_num_parameters(model, count_nonzero_only=True)
    # Print all measurment
    print(f"Our model with {l2_lambda} regularization has accuracy on test set={test_acc:.2f}%")
    print(f"Our model with {l2_lambda} regularization has size={model_size/MiB:.2f} MiB")
    print(f"The time inference of our model with {l2_lambda} regularization is ={t_inference}") 
    print(f"The number of parametrs of our model with {l2_lambda} regularization is:{num_parm}")



    #Update my Eva dictionary
    Eva.update({'Model accuracy': test_acc,
                'time inference': t_inference,
                'number parmameters of model': num_parm,
                'size of model': model_size})


    Accuracy.append(Eva['Model accuracy'])
    T_inference.append(Eva['time inference'])
    Num_parm.append(int(Eva['number parmameters of model']))
    Model_size.append(int(Eva['size of model']))

_________________________________________________________
This is step:1
Epoch: 010,Train loss:0.7127, Val: 0.9069, Test: 0.8760
Epoch: 020,Train loss:0.3721, Val: 0.9508, Test: 0.8820
Epoch: 030,Train loss:0.3294, Val: 0.9737, Test: 0.9120
Epoch: 040,Train loss:0.3262, Val: 0.9731, Test: 0.8920
Our model with 0.0001 regularization has accuracy on test set=0.90%
Our model with 0.0001 regularization has size=0.26 MiB
The time inference of our model with 0.0001 regularization is =29.325424671173096
The number of parametrs of our model with 0.0001 regularization is:69143
_________________________________________________________
This is step:2
Epoch: 010,Train loss:0.7267, Val: 0.9040, Test: 0.8640
Epoch: 020,Train loss:0.4066, Val: 0.9420, Test: 0.8840
Epoch: 030,Train loss:0.3522, Val: 0.9655, Test: 0.8860
Epoch: 040,Train loss:0.3162, Val: 0.9684, Test: 0.9000
Our model with 0.0001 regularization has accuracy on test set=0.90%
Our model with 0.0001 regularization has size=0.26 MiB
The t

In [9]:
Eva_final=dict()
std=dict()
print(100 * "=")

model_accuracy_mean = stat.mean(Accuracy)
model_accuracy_std =  stat.stdev(Accuracy)
Eva_final.update({'model accuracy':float(format(model_accuracy_mean, '.4f'))})
std.update({'Std of base model accuracy':float(format(model_accuracy_std, '.4f'))})

desc_model_accuracy = "{:.3f} ± {:.3f}".format(model_accuracy_mean,model_accuracy_std)
print(f"The mean of model accuracy:{desc_model_accuracy}")


print(100 * "=")
####Time inference    
t_model_mean =stat.mean(T_inference)
t_model_std =stat.stdev(T_inference)
Eva_final.update({'time inference of model':float(format(t_model_mean, '.6f'))})
std.update({'Std of time inference of model':float(format(t_model_std, '.6f'))})

desc_t_model = "{:.3f} ± {:.3f}".format(t_model_mean,t_model_std)
print(f"The mean of inference time is :{desc_t_model} ")

print(100 * "=")
####Number of Parameters   
num_parm_model_mean = stat.mean(Num_parm)
Eva_final.update({'number parmameters of model':num_parm_model_mean})
print(f"The number of parameters is :{num_parm_model_mean} ")

print(100 * "=")
####Model Size 
model_size_mean = stat.mean(Model_size)
Eva_final.update({'base_model_size':model_size_mean})
print(f"The model size is :{model_size_mean} ")



The mean of model accuracy:0.899 ± 0.003
The mean of inference time is :28.867 ± 0.475 
The number of parameters is :69143 
The model size is :2212576 


In [10]:
Eva_final

{'model accuracy': 0.8987,
 'time inference of model': 28.867402,
 'number parmameters of model': 69143,
 'base_model_size': 2212576}

In [11]:
Cora_Node_00001=Eva_final
print(Cora_Node_00001)

{'model accuracy': 0.8987, 'time inference of model': 28.867402, 'number parmameters of model': 69143, 'base_model_size': 2212576}


In [12]:
Cora_std_00001=std
print(Cora_std_00001)

{'Std of base model accuracy': 0.0031, 'Std of time inference of model': 0.475353}
